<a href="https://colab.research.google.com/github/GarzonDiegoINL/Characterization/blob/main/EIS_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# EIS Analysis — BioLogic `.mpr` Files

**Workflow:**
1. Load binary `.mpr` data via `galvani` (same as the NOMAD plugin)
2. Optionally extract metadata via `yadg`
3. Preprocess data (remove inductive artefacts)
4. Validate with Kramers-Kronig
5. Fit a multi-RC equivalent circuit with bounded CPE elements using `impedance.py`
6. Generate publication-quality Nyquist + residual plots

**Circuit string syntax** (impedance.py):  
- 2 arcs: `R0-p(R1,CPE1)-p(R2,CPE2)`  
- 3 arcs: `R0-p(R1,CPE1)-p(R2,CPE2)-p(R3,CPE3)`  
- `R` = resistor, `CPE(Q, α)` = Constant Phase Element where $Z = \frac{1}{Q(j2\pi f)^\alpha}$

In [ ]:
!pip install galvani impedance yadg

In [ ]:
from __future__ import annotations

import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.ticker import LogLocator, LogFormatter

from galvani import MPRfile
from impedance.models.circuits import CustomCircuit
from impedance import preprocessing
from impedance.validation import linKK

warnings.filterwarnings("ignore", category=RuntimeWarning)

plt.rcParams.update({
    "font.size": 12,
    "axes.labelsize": 13,
    "axes.titlesize": 13,
    "legend.fontsize": 11,
    "figure.dpi": 120,
})

print("All imports OK.")

---
## 1 — Data Loading

The `load_mpr` function mirrors the NOMAD plugin's `MPRParser`:
- **`galvani`** reads the binary `.mpr` measurement data (columns depend on technique; EIS uses `freq/Hz`, `Re(Z)/Ohm`, `-Im(Z)/Ohm`)
- **`yadg`** (optional) extracts the settings block — electrode area, material, electrolyte, comments

The complex impedance is assembled as  
$$Z = \text{Re}(Z) - j \cdot \bigl(-\text{Im}(Z)\bigr)$$  
because BioLogic EC-Lab stores the sign-flipped imaginary part as the column `-Im(Z)/Ohm`.

In [ ]:
from __future__ import annotations


def load_mpr(filepath: str | Path) -> tuple[np.ndarray, np.ndarray, pd.DataFrame, dict]:
    """Load a BioLogic .mpr EIS file.

    Uses `galvani` for measurement data (same as the NOMAD MPRParser plugin)
    and `yadg` (optional) for the settings metadata block.

    Parameters
    ----------
    filepath : str or Path
        Path to the `.mpr` binary file.

    Returns
    -------
    frequencies : np.ndarray
        Frequency array in Hz (descending, as measured).
    Z : np.ndarray of dtype complex128
        Complex impedance array.  Re(Z) - j*Im(Z) in Ohm.
    df_raw : pd.DataFrame
        Raw DataFrame from galvani containing all logged columns.
    metadata : dict
        Settings extracted by yadg (empty dict if yadg is unavailable or fails).
    """
    filepath = Path(filepath)

    # ── 1. Measurement data via galvani ──────────────────────────────────────
    mpr = MPRfile(str(filepath))
    df_raw = pd.DataFrame(mpr.data)

    required = {"freq/Hz", "Re(Z)/Ohm", "-Im(Z)/Ohm"}
    missing = required - set(df_raw.columns)
    if missing:
        raise ValueError(
            f"File does not appear to be an EIS measurement. "
            f"Missing columns: {missing}\n"
            f"Available columns: {list(df_raw.columns)}"
        )

    frequencies = df_raw["freq/Hz"].to_numpy(dtype=np.float64)
    # BioLogic stores -Im(Z), so: Z = Re - j*(-Im(Z)/Ohm) = Re + j*Im
    # But impedance.py convention: Im(Z) < 0 for capacitive → Z = Re - j*|Im|
    Z = (
        df_raw["Re(Z)/Ohm"].to_numpy(dtype=np.float64)
        - 1j * df_raw["-Im(Z)/Ohm"].to_numpy(dtype=np.float64)
    )

    # ── 2. Metadata via yadg (optional) ──────────────────────────────────────
    metadata: dict = {}
    try:
        import yadg

        dt = yadg.extractors.extract(
            filetype="eclab.mpr", path=filepath, timezone="UTC"
        )
        raw_meta = json.loads(dt.attrs.get("original_metadata", "{}"))
        settings = raw_meta.get("settings", raw_meta)
        for key in (
            "electrode_area",
            "electrode_material",
            "electrolyte",
            "reference_electrode",
            "comments",
        ):
            val = settings.get(key)
            if val:
                metadata[key] = val
    except Exception as exc:
        print(f"[yadg] metadata extraction skipped: {exc}")

    return frequencies, Z, df_raw, metadata

---
## 2 — Configuration

**Edit this cell** to match your measurement.

### Parameter ordering rules
`impedance.py` reads parameters **left-to-right** through the circuit string.  
For `R0-p(R1,CPE1)-p(R2,CPE2)` the full order is:

| Index | Name    | Description                      | Typical range         |
|-------|---------|----------------------------------|-----------------------|
| 0     | `R0`    | Series / contact resistance      | 1 – 1 000 Ω           |
| 1     | `R1`    | Arc 1 resistance                 | 10 – 1 × 10⁶ Ω        |
| 2     | `CPE1_0`| Arc 1 CPE prefactor Q            | 1×10⁻¹⁰ – 1×10⁻³      |
| 3     | `CPE1_1`| Arc 1 CPE exponent α             | 0.5 – 1.0             |
| 4     | `R2`    | Arc 2 resistance                 | 10 – 1 × 10⁶ Ω        |
| 5     | `CPE2_0`| Arc 2 CPE prefactor Q            | 1×10⁻¹⁰ – 1×10⁻³      |
| 6     | `CPE2_1`| Arc 2 CPE exponent α             | 0.5 – 1.0             |

> **Freezing parameters**: add an entry to `FROZEN_PARAMS`, e.g. `{"R0": 10.0}`.  
> The `INITIAL_GUESS` and bounds arrays below cover **all** parameters for reference,  
> but only the **free** ones (those not in `FROZEN_PARAMS`) are passed to the fitter.

In [ ]:
# ═══════════════════════════════════════════════════════════════════
#  USER CONFIGURATION — edit values in this cell
# ═══════════════════════════════════════════════════════════════════

# ── File (ignored in Colab — the upload cell sets FILE_PATH automatically) ───
FILE_PATH = "your_file.mpr"    # ← set your local path here

# ── Circuit topology ─────────────────────────────────────────────────────────
N_RC = 2   # number of RC arcs: 2 or 3

_arcs = "-".join(f"p(R{i},CPE{i})" for i in range(1, N_RC + 1))
CIRCUIT_STRING = f"R0-{_arcs}"
print(f"Circuit: {CIRCUIT_STRING}")

# ── SPEIS: which voltage steps to fit ────────────────────────────────────────
# "all"   → fit every cycle found in the file
# [1,3,5] → fit only those cycle numbers (e.g. [2, 4, 6, 8, 10, 12])
CYCLES_TO_FIT = "all"

# ── Frequency window (Hz) — set None to keep all data ────────────────────────
F_MIN = None    # e.g. 1e-2
F_MAX = None    # e.g. 1e6

# ── Parameters to freeze (held constant during fit) ──────────────────────────
# Leave empty {} to fit all parameters freely.
# Example: FROZEN_PARAMS = {"R0": 10.0}  freezes series resistance at 10 Ω
FROZEN_PARAMS: dict[str, float] = {}

# ── Initial guesses (ALL parameters, in circuit order) ───────────────────────
# 2-arc order: R0, R1, CPE1_0, CPE1_1, R2, CPE2_0, CPE2_1
# 3-arc order: add R3, CPE3_0, CPE3_1
if N_RC == 2:
    INITIAL_GUESS_ALL = [
        50.0,      # R0   — series resistance [Ω]
        500.0,     # R1   — arc 1 resistance  [Ω]
        1e-6,      # CPE1_0  Q [S·sᵅ]
        0.85,      # CPE1_1  α (0.5–1.0)
        2000.0,    # R2   — arc 2 resistance  [Ω]
        1e-7,      # CPE2_0  Q [S·sᵅ]
        0.80,      # CPE2_1  α (0.5–1.0)
    ]
elif N_RC == 3:
    INITIAL_GUESS_ALL = [
        50.0, 500.0, 1e-6, 0.85,
        2000.0, 1e-7, 0.80,
        5000.0, 1e-8, 0.75,
    ]
else:
    raise ValueError("N_RC must be 2 or 3")

# ── Bounds (ALL parameters, same order as INITIAL_GUESS_ALL) ─────────────────
# Strict CPE alpha limits: 0.5 ≤ α ≤ 1.0
if N_RC == 2:
    LOWER_BOUNDS_ALL = [0.0,  0.0,  1e-12, 0.5,  0.0,  1e-12, 0.5]
    UPPER_BOUNDS_ALL = [1e4,  1e7,  1e-1,  1.0,  1e7,  1e-1,  1.0]
elif N_RC == 3:
    LOWER_BOUNDS_ALL = [0.0,  0.0,  1e-12, 0.5,  0.0,  1e-12, 0.5,  0.0,  1e-12, 0.5]
    UPPER_BOUNDS_ALL = [1e4,  1e7,  1e-1,  1.0,  1e7,  1e-1,  1.0,  1e7,  1e-1,  1.0]

---
## 3 — Load Data & Preview

In [ ]:
# ── Google Colab file upload ─────────────────────────────────────────────────
# When running in Colab, this cell shows a file-picker widget and sets FILE_PATH
# automatically.  Running locally, it just confirms the path from the config cell.
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import files as _colab_files
    print("Running in Google Colab — please upload your .mpr file:")
    _uploaded = _colab_files.upload()        # shows the upload widget
    if _uploaded:
        FILE_PATH = list(_uploaded.keys())[0]
        print(f"\n✔  FILE_PATH set to: '{FILE_PATH}'")
    else:
        print("⚠  No file uploaded. Set FILE_PATH manually in the configuration cell.")
else:
    print(f"Running locally.  FILE_PATH = '{FILE_PATH}'")
    print("(In Google Colab this cell will display a file-upload widget.)")

In [ ]:
_, _, df_raw, metadata = load_mpr(FILE_PATH)

# ── Print metadata ────────────────────────────────────────────────────────────
print("═" * 56)
print(f"  File : {Path(FILE_PATH).name}")
if metadata:
    print("  Metadata from yadg:")
    for k, v in metadata.items():
        print(f"    {k}: {v}")
else:
    print("  No yadg metadata.")
print("═" * 56)

# ── Detect single EIS vs SPEIS (multiple voltage steps) ──────────────────────
if "cycle number" in df_raw.columns and df_raw["cycle number"].nunique() > 1:
    IS_SPEIS = True
    cycles_in_file = sorted(df_raw["cycle number"].unique())
    spectra_all = {}
    for cyc in cycles_in_file:
        sub = df_raw[df_raw["cycle number"] == cyc]
        freq = sub["freq/Hz"].to_numpy(np.float64)
        Z = (sub["Re(Z)/Ohm"].to_numpy(np.float64)
             - 1j * sub["-Im(Z)/Ohm"].to_numpy(np.float64))
        valid = freq > 0          # drop any bookkeeping rows with f = 0
        freq, Z = freq[valid], Z[valid]
        voltage = float(sub["<Ewe>/V"].mean()) if "<Ewe>/V" in sub.columns else np.nan
        spectra_all[cyc] = {"freq": freq, "Z": Z, "voltage": voltage}

    print(f"SPEIS file detected — {len(spectra_all)} voltage steps:")
    print(f"  {'Cycle':>7}  {'Voltage (V)':>12}  {'Points':>7}")
    for cyc, sp in spectra_all.items():
        print(f"  {cyc:>7.0f}  {sp['voltage']:>12.4f}  {len(sp['freq']):>7}")
else:
    IS_SPEIS = False
    freq_all = df_raw["freq/Hz"].to_numpy(np.float64)
    Z_all = (df_raw["Re(Z)/Ohm"].to_numpy(np.float64)
             - 1j * df_raw["-Im(Z)/Ohm"].to_numpy(np.float64))
    spectra_all = {1: {"freq": freq_all, "Z": Z_all, "voltage": np.nan}}
    print(f"Single EIS spectrum — {len(freq_all)} points,  "
          f"f = {freq_all.min():.3g} – {freq_all.max():.3g} Hz")

In [ ]:
# ── Overview Nyquist: all raw spectra overlaid, coloured by voltage ───────────
v_vals_all = [sp["voltage"] for sp in spectra_all.values()]
cmap_ov = plt.cm.plasma
v_lo, v_hi = min(v_vals_all), max(v_vals_all)
norm_ov = plt.Normalize(v_lo, v_hi) if v_lo != v_hi else plt.Normalize(v_lo - 1, v_hi + 1)

fig, ax = plt.subplots(figsize=(7, 6))
for cyc, sp in spectra_all.items():
    color = cmap_ov(norm_ov(sp["voltage"])) if np.isfinite(sp["voltage"]) else "steelblue"
    ax.plot(sp["Z"].real, -sp["Z"].imag, "-", lw=1.2, color=color, alpha=0.8)

sm = plt.cm.ScalarMappable(cmap=cmap_ov, norm=norm_ov)
sm.set_array([])
fig.colorbar(sm, ax=ax, label="DC Bias (V)")
ax.set_xlabel("Re(Z) / Ω", fontsize=13)
ax.set_ylabel("−Im(Z) / Ω", fontsize=13)
ax.set_title(f"All spectra — {Path(FILE_PATH).stem}", fontsize=13)
ax.grid(True, linestyle="--", alpha=0.35)
plt.tight_layout()
plt.show()

---
## 4 — Preprocessing

Two filters are applied:

1. **`ignoreBelowX`** — removes data points where Re(Z) < 0, which arise from high-frequency inductive artefacts (parasitic wiring inductance).
2. **Frequency crop** — optional window defined by `F_MIN` / `F_MAX` in the configuration cell.

The Nyquist scatter below helps judge whether the raw data is clean.

In [ ]:
# ── Select which cycles to process ───────────────────────────────────────────
if CYCLES_TO_FIT == "all":
    selected_cycles = list(spectra_all.keys())
else:
    selected_cycles = [c for c in CYCLES_TO_FIT if c in spectra_all]
    missing = [c for c in CYCLES_TO_FIT if c not in spectra_all]
    if missing:
        print(f"Warning: cycles not found in file: {missing}")

# ── Apply filters to every selected spectrum ──────────────────────────────────
clean_spectra = {}
for cyc in selected_cycles:
    sp = spectra_all[cyc]
    freq, Z = sp["freq"], sp["Z"]
    n_raw_sp = len(freq)

    # Step 1: remove inductive artefacts (Re(Z) < 0)
    freq_f, Z_f = preprocessing.ignoreBelowX(freq, Z)

    # Step 2: optional frequency crop
    if F_MIN is not None or F_MAX is not None:
        f_lo = F_MIN if F_MIN is not None else 0.0
        f_hi = F_MAX if F_MAX is not None else np.inf
        mask = (freq_f >= f_lo) & (freq_f <= f_hi)
        freq_f, Z_f = freq_f[mask], Z_f[mask]

    clean_spectra[cyc] = {"freq": freq_f, "Z": Z_f, "voltage": sp["voltage"]}
    v_str = f"{sp['voltage']:+.4f} V" if np.isfinite(sp["voltage"]) else "N/A"
    print(f"  Cycle {cyc:.0f}  ({v_str}):  "
          f"{n_raw_sp} → {len(freq_f)} pts  (removed {n_raw_sp - len(freq_f)})")

print(f"\nReady to fit {len(clean_spectra)} spectra.")

# ── Preprocessed Nyquist preview (overlaid) ───────────────────────────────────
v_vals_c = [sp["voltage"] for sp in clean_spectra.values()]
norm_c = plt.Normalize(min(v_vals_c), max(v_vals_c)) if min(v_vals_c) != max(v_vals_c) \
         else plt.Normalize(min(v_vals_c) - 1, max(v_vals_c) + 1)
cmap_c = plt.cm.plasma

fig, ax = plt.subplots(figsize=(7, 6))
for cyc, sp in clean_spectra.items():
    color = cmap_c(norm_c(sp["voltage"])) if np.isfinite(sp["voltage"]) else "steelblue"
    ax.plot(sp["Z"].real, -sp["Z"].imag, "o-", ms=3, lw=1, color=color, alpha=0.8)

sm2 = plt.cm.ScalarMappable(cmap=cmap_c, norm=norm_c)
sm2.set_array([])
fig.colorbar(sm2, ax=ax, label="DC Bias (V)")
ax.set_xlabel("Re(Z) / Ω", fontsize=13)
ax.set_ylabel("−Im(Z) / Ω", fontsize=13)
ax.set_title("Preprocessed spectra", fontsize=13)
ax.grid(True, linestyle="--", alpha=0.35)
plt.tight_layout()
plt.show()

---
## 5 — Kramers-Kronig Validation

The **Linear Kramers-Kronig (Lin-KK)** test checks whether the data is causal, linear, and stable — the three prerequisites for valid EIS. Residuals should be flat and within ±2 % across the full frequency range. Larger systematic deviations indicate drift, non-linearity, or artefacts.

In [ ]:
# Workaround for impedance.py ≤1.7.1 bug: np is not in eval_linKK's namespace
import impedance.validation as _iv
from impedance.models.circuits.elements import circuit_elements as _ce
_ce["np"] = np

M, mu, Z_linKK, res_real, res_imag = linKK(
    frequencies_clean, Z_clean, c=0.85, max_M=50, fit_type="complex", add_cap=True
)

print(f"Lin-KK: M = {M},  μ = {mu:.4f}  (< 0.5 = good)")

fig, ax = plt.subplots(figsize=(7, 3.5))
ax.semilogx(frequencies_clean, res_real * 100, "o-", ms=4, lw=1.2,
            color="steelblue", label=r"$\Delta_\mathrm{Re}$")
ax.semilogx(frequencies_clean, res_imag * 100, "s--", ms=4, lw=1.2,
            color="tomato", label=r"$\Delta_\mathrm{Im}$")
ax.axhline(0, color="black", lw=0.8)
ax.axhspan(-2, 2, alpha=0.08, color="green", label="±2 % band")
ax.set_xlabel("Frequency / Hz")
ax.set_ylabel("Residual / %")
ax.set_title("Lin-KK Residuals")
ax.legend(loc="upper right")
ax.grid(True, which="both", linestyle="--", alpha=0.4)
ax.set_ylim(-10, 10)
plt.tight_layout()
plt.show()

---
## 6 — Equivalent Circuit Fitting

### What happens here
1. The parameter names are queried with `get_param_names()` to confirm the ordering.
2. Any parameters listed in `FROZEN_PARAMS` are removed from `initial_guess` and `bounds` — only **free** parameters are passed to `CustomCircuit`.
3. `circuit.fit()` is called with:
   - `bounds` — enforces physical constraints (CPE α strictly in [0.5, 1.0])
   - `weight_by_modulus=True` — uses |Z| as weights, the standard approach when experimental variances are unknown
4. Fit parameters and their 1σ errors are printed.

In [ ]:
# ── Build a temporary circuit (no constants) to get the parameter name list ──
_tmp = CustomCircuit(CIRCUIT_STRING, initial_guess=INITIAL_GUESS_ALL)
all_param_names, all_param_units = _tmp.get_param_names()

print("All parameters in circuit order:")
for name, unit, val, lo, hi in zip(
    all_param_names, all_param_units,
    INITIAL_GUESS_ALL, LOWER_BOUNDS_ALL, UPPER_BOUNDS_ALL,
):
    frozen_tag = f"  ← FROZEN at {FROZEN_PARAMS[name]}" if name in FROZEN_PARAMS else ""
    print(f"  {name:10s}  [{unit}]   init={val:.2e}   bounds=[{lo:.1e}, {hi:.1e}]{frozen_tag}")

# ── Filter to free parameters only ────────────────────────────────────────────
free_names   = [n for n in all_param_names if n not in FROZEN_PARAMS]
free_indices = [i for i, n in enumerate(all_param_names) if n not in FROZEN_PARAMS]

initial_guess_free = [INITIAL_GUESS_ALL[i] for i in free_indices]
lower_bounds_free  = [LOWER_BOUNDS_ALL[i]  for i in free_indices]
upper_bounds_free  = [UPPER_BOUNDS_ALL[i]  for i in free_indices]

assert len(initial_guess_free) == len(lower_bounds_free) == len(upper_bounds_free), \
    "Mismatch between initial_guess and bounds lengths — check N_RC."

print(f"\nFree parameters ({len(free_names)}): {free_names}")
print(f"Frozen parameters: {list(FROZEN_PARAMS.keys())}")

In [ ]:
# ── Instantiate and fit ───────────────────────────────────────────────────────
circuit = CustomCircuit(
    CIRCUIT_STRING,
    initial_guess=initial_guess_free,
    constants=FROZEN_PARAMS,
)

circuit.fit(
    frequencies_clean,
    Z_clean,
    bounds=(lower_bounds_free, upper_bounds_free),
    weight_by_modulus=True,
)

# ── Print fitted parameters ───────────────────────────────────────────────────
param_names_fit, param_units_fit = circuit.get_param_names()
params = circuit.parameters_
errors = circuit.conf_

print("\n" + "═" * 58)
print(f"  Fitted circuit: {CIRCUIT_STRING}")
print("═" * 58)
print(f"  {'Parameter':<12} {'Value':>14}  {'±1σ':>14}  Unit")
print("  " + "-" * 54)
for name, unit, val, err in zip(param_names_fit, param_units_fit, params, errors):
    print(f"  {name:<12} {val:>14.4e}  {err:>14.4e}  {unit}")
print("═" * 58)

# Relative error check (>50 % → poor convergence warning)
for name, val, err in zip(param_names_fit, params, errors):
    if err / abs(val) > 0.5:
        print(f"  ⚠  {name}: large relative error ({err/abs(val)*100:.0f} %) — "
              "consider tightening bounds or adjusting initial guess.")

---
## 7 — Visualization

### 7a — Nyquist plot with fit residuals

The figure has two panels:
- **Top (Nyquist):** experimental data as filled circles, fitted model as a solid line. The frequencies are annotated on a subset of points (decade steps).
- **Bottom (Residuals):** normalised residuals in % for the real and imaginary parts vs log-frequency. A good fit stays within ±5 %.

In [ ]:
def plot_nyquist_with_residuals(
    freq: np.ndarray,
    Z_data: np.ndarray,
    Z_fit: np.ndarray,
    title: str = "",
    save_path: str | None = None,
) -> None:
    """Publication-quality Nyquist plot with normalised residual panel.

    Parameters
    ----------
    freq     : frequency array (Hz)
    Z_data   : experimental complex impedance
    Z_fit    : fitted complex impedance (same length as freq)
    title    : optional suptitle
    save_path: if given, save figure to this path (e.g. "nyquist.pdf")
    """
    # Normalised residuals (%)
    norm = np.abs(Z_data)
    res_re = (Z_fit.real - Z_data.real) / norm * 100
    res_im = (Z_fit.imag - Z_data.imag) / norm * 100

    # Use constrained_layout to avoid tight_layout conflict with equal-aspect axes
    fig = plt.figure(figsize=(7, 8), constrained_layout=True)
    gs = gridspec.GridSpec(2, 1, height_ratios=[3, 1], figure=fig, hspace=0.08)

    # ── Top: Nyquist ─────────────────────────────────────────────────────────
    ax_ny = fig.add_subplot(gs[0])
    ax_ny.scatter(
        Z_data.real, -Z_data.imag,
        s=45, zorder=3, color="steelblue", edgecolors="white", linewidths=0.5,
        label="Experiment",
    )
    ax_ny.plot(
        Z_fit.real, -Z_fit.imag,
        lw=2, color="tomato", zorder=2, label=f"Fit  ({CIRCUIT_STRING})",
    )

    # Annotate decade frequency markers — skip if annotation would overlap previous one
    log_f = np.log10(freq)
    decade_targets = np.arange(np.ceil(log_f.min()), np.floor(log_f.max()) + 1)
    # Compute data range to set a minimum pixel-space distance threshold
    x_range = Z_data.real.max() - Z_data.real.min()
    min_spacing = x_range * 0.12     # skip annotation if too close to previous
    last_xy = (-np.inf, -np.inf)
    for target in decade_targets:
        idx = np.argmin(np.abs(log_f - target))
        xy = (Z_data.real[idx], -Z_data.imag[idx])
        dist = np.hypot(xy[0] - last_xy[0], xy[1] - last_xy[1])
        if dist < min_spacing:
            continue
        ax_ny.annotate(
            f"{10**target:.0e} Hz",
            xy=xy,
            xytext=(5, 5), textcoords="offset points",
            fontsize=8, color="dimgray",
        )
        last_xy = xy

    ax_ny.set_ylabel("−Im(Z) / Ω", fontsize=13)
    ax_ny.set_aspect("equal", adjustable="datalim")
    ax_ny.legend(fontsize=11, loc="upper left")
    ax_ny.grid(True, linestyle="--", alpha=0.35)
    if title:
        ax_ny.set_title(title, fontsize=13, pad=8)

    # ── Bottom: Residuals ─────────────────────────────────────────────────────
    ax_res = fig.add_subplot(gs[1])
    ax_res.semilogx(
        freq, res_re, "o-", ms=4, lw=1.2, color="steelblue", label=r"$\Delta_\mathrm{Re}$"
    )
    ax_res.semilogx(
        freq, res_im, "s--", ms=4, lw=1.2, color="tomato", label=r"$\Delta_\mathrm{Im}$"
    )
    ax_res.axhline(0, color="black", lw=0.8)
    ax_res.axhspan(-5, 5, alpha=0.07, color="green", label="±5 % band")
    ax_res.set_xlabel("Frequency / Hz", fontsize=13)
    ax_res.set_ylabel("Residual / %", fontsize=11)
    ax_res.legend(fontsize=9, loc="upper right", ncol=3)
    ax_res.grid(True, which="both", linestyle="--", alpha=0.35)
    ax_res.set_ylim(-15, 15)

    if save_path:
        fig.savefig(save_path, dpi=200, bbox_inches="tight")
        print(f"Saved → {save_path}")
    plt.show()


# ── Generate fitted impedance and call plot ───────────────────────────────────
Z_fit = circuit.predict(frequencies_clean)

plot_nyquist_with_residuals(
    frequencies_clean,
    Z_clean,
    Z_fit,
    title=Path(FILE_PATH).stem,
    save_path=None,      # e.g. "nyquist_fit.pdf"
)

### 7b — Bode plot

In [ ]:
fig, (ax_mod, ax_ph) = plt.subplots(2, 1, figsize=(7, 6), sharex=True)

# |Z| vs frequency
ax_mod.loglog(frequencies_clean, np.abs(Z_clean), "o", ms=5, color="steelblue",
              label="Experiment", zorder=3)
ax_mod.loglog(frequencies_clean, np.abs(Z_fit), "-", lw=2, color="tomato",
              label="Fit", zorder=2)
ax_mod.set_ylabel(r"$|Z|$ / Ω", fontsize=13)
ax_mod.legend(fontsize=11)
ax_mod.grid(True, which="both", linestyle="--", alpha=0.35)
ax_mod.set_title(f"Bode plot — {Path(FILE_PATH).stem}", fontsize=13)

# Phase vs frequency
phase_data = np.angle(Z_clean, deg=True)
phase_fit  = np.angle(Z_fit, deg=True)
ax_ph.semilogx(frequencies_clean, phase_data, "o", ms=5, color="steelblue", zorder=3)
ax_ph.semilogx(frequencies_clean, phase_fit, "-", lw=2, color="tomato", zorder=2)
ax_ph.set_xlabel("Frequency / Hz", fontsize=13)
ax_ph.set_ylabel("Phase / °", fontsize=13)
ax_ph.grid(True, which="both", linestyle="--", alpha=0.35)

plt.tight_layout()
plt.show()

---
## 8 — Parameter Summary & Derived Quantities

For each RC arc $i$ the following quantities are derived from the CPE parameters:

| Quantity | Formula |
|----------|---------|
| Effective capacitance | $C_{\text{eff},i} = Q_i^{1/\alpha_i} \cdot R_i^{(1-\alpha_i)/\alpha_i}$ |
| Time constant | $\tau_i = R_i \cdot C_{\text{eff},i} = R_i^{1/\alpha_i} \cdot Q_i^{1/\alpha_i}$ |

These are the standard conversions from the Hahn–Bard (2001) formulation.

In [ ]:
# ── Fitted parameter table ───────────────────────────────────────────────────
rows = []
param_names_fit, param_units_fit = circuit.get_param_names()
for name, unit, val, err in zip(param_names_fit, param_units_fit, params, errors):
    rows.append({
        "Parameter": name,
        "Value":     val,
        "±1σ Error": err,
        "Rel. Error (%)": f"{err/abs(val)*100:.1f}" if val != 0 else "—",
        "Unit": unit,
    })

# Add frozen parameters as reference rows
for name, val in FROZEN_PARAMS.items():
    rows.append({
        "Parameter": name,
        "Value":     val,
        "±1σ Error": 0.0,
        "Rel. Error (%)": "FROZEN",
        "Unit": "Ohm",
    })

df_params = pd.DataFrame(rows)
display(df_params.set_index("Parameter").style.format(
    {"Value": "{:.4e}", "±1σ Error": "{:.4e}"}
))

# ── Derived quantities per arc ───────────────────────────────────────────────
# Collect all fitted + frozen params in one dict for easy lookup
all_fitted: dict[str, float] = dict(zip(param_names_fit, params))
all_fitted.update(FROZEN_PARAMS)

print("\nDerived quantities per RC arc:")
print(f"  {'Arc':<6}  {'R [Ω]':>12}  {'Q [S·sᵅ]':>12}  {'α':>6}  {'C_eff [F]':>12}  {'τ [s]':>12}  {'f_peak [Hz]':>12}")
print("  " + "-" * 74)

for i in range(1, N_RC + 1):
    R_key  = f"R{i}"
    Q_key  = f"CPE{i}_0"
    a_key  = f"CPE{i}_1"

    R = all_fitted.get(R_key)
    Q = all_fitted.get(Q_key)
    a = all_fitted.get(a_key)

    if None in (R, Q, a):
        continue

    C_eff = Q ** (1.0 / a) * R ** ((1.0 - a) / a)
    tau   = R * C_eff
    f_peak = 1.0 / (2 * np.pi * tau)

    print(f"  Arc {i}   {R:>12.3e}  {Q:>12.3e}  {a:>6.3f}  {C_eff:>12.3e}  {tau:>12.3e}  {f_peak:>12.3e}")

In [ ]:
# ── Fitted parameters vs DC Bias ──────────────────────────────────────────────
voltages_fit = df_summary["Voltage (V)"].values.astype(float)
pnames_to_plot = list(param_names_fit)

ncols_p = 2
nrows_p = int(np.ceil(len(pnames_to_plot) / ncols_p))

fig, axes_p = plt.subplots(nrows_p, ncols_p,
                            figsize=(ncols_p * 5, nrows_p * 3.5),
                            constrained_layout=True)
axes_p_flat = axes_p.flatten() if hasattr(axes_p, "flatten") else [axes_p]

for ax_p, pname in zip(axes_p_flat, pnames_to_plot):
    vals = df_summary[pname].values
    err_col = f"{pname}_err"
    errs_p = df_summary[err_col].values if err_col in df_summary.columns \
             else np.zeros(len(vals))
    ax_p.errorbar(voltages_fit, vals, yerr=errs_p,
                  fmt="o-", ms=6, lw=1.5, capsize=4)
    ax_p.set_xlabel("Voltage (V)", fontsize=11)
    ax_p.set_ylabel(pname, fontsize=11)
    ax_p.set_title(pname, fontsize=12)
    ax_p.grid(True, linestyle="--", alpha=0.35)

for ax_p in axes_p_flat[len(pnames_to_plot):]:
    ax_p.set_visible(False)

fig.suptitle(f"Fitted parameters vs DC Bias — {Path(FILE_PATH).stem}", fontsize=13)
plt.show()

---
## 9 — Save Results

- **`circuit.save()`** exports the model (circuit string + fitted parameters) as JSON. Reload with `circuit.load()`.
- The parameter table is saved as CSV alongside the model JSON.

In [ ]:
stem = Path(FILE_PATH).stem

# ── Save impedance.py model as JSON ──────────────────────────────────────────
model_path = f"{stem}_eis_model.json"
circuit.save(model_path)
print(f"Model saved → {model_path}")
print("  Reload with: circuit = CustomCircuit(''); circuit.load(model_path)")

# ── Save parameter table as CSV ───────────────────────────────────────────────
csv_path = f"{stem}_eis_params.csv"
df_params.to_csv(csv_path, index=False)
print(f"Parameters saved → {csv_path}")

# ── Save clean data + fit as CSV ─────────────────────────────────────────────
df_fit = pd.DataFrame({
    "freq/Hz":          frequencies_clean,
    "Re(Z)_exp/Ohm":    Z_clean.real,
    "-Im(Z)_exp/Ohm":  -Z_clean.imag,
    "Re(Z)_fit/Ohm":    Z_fit.real,
    "-Im(Z)_fit/Ohm":  -Z_fit.imag,
})
data_path = f"{stem}_eis_fit.csv"
df_fit.to_csv(data_path, index=False)
print(f"Data + fit saved → {data_path}")